In [2]:
!pip install remfile dandi pynwb h5py scikit-learn matplotlib numpy

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ---------------------------------------- 0.0/832.0 kB ? eta -:--:--
   ---------------------------------------- 832.0/832.0 kB 6.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   --------------- ------------------------ 0.5/1.4 MB 3.5 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 3.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/42.3 MB ? eta -:--:--
   ---------------------------------------- 0.5/42.3 MB 4.4 MB/s eta 0:00:10
   - -------------------------------------- 1.6/42.3 MB 4.0 MB/s eta 0:00:11
   - -------------------------------------- 2.1/42.3 MB 3.9 MB/s eta 0:00:11
   --- ------------------------------------ 3.4/42.3 MB 4.1 MB/s eta 0:00:10
   --- ------------------------------------ 3.9/42.3 MB 3.8 MB/s eta 0:00:10
   ---- ----------------------------------- 4.7/42.3 MB 4.0 MB/s eta 0:

  DEPRECATION: Building 'asciitree' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'asciitree'. Discussion can be found at https://github.com/pypa/pip/issues/6334


In [3]:
import numpy as np
import matplotlib.pyplot as plt
import remfile
import h5py
from pynwb import NWBHDF5IO
from dandi.dandiapi import DandiAPIClient
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import time

# ============================================================================
# Connecting to dataset
# ============================================================================

def connect_to_data():
    """Connect to the dataset and return the nwb file."""
    client = DandiAPIClient()
    dandiset = client.get_dandiset("000402", "draft")
    all_assets = dandiset.get_assets()

    asset = None
    for a in all_assets:
        if a.path.endswith(".nwb"):
            asset = a
            break

    s3_url = asset.get_content_url(follow_redirects=1, strip_query=True)
    rf = remfile.File(s3_url)
    h5 = h5py.File(rf, "r")
    io = NWBHDF5IO(file=h5, load_namespaces=True)
    nwb = io.read()

    return nwb


def get_all_roi_response_series(nwb):
    """Get all available ROI response series from the NWB file."""
    ophys = nwb.processing["ophys"]
    fluorescence = ophys.data_interfaces["Fluorescence"]
    
    all_series = {}
    for key in fluorescence.roi_response_series.keys():
        all_series[key] = fluorescence.roi_response_series[key]
    
    return all_series


def get_stimulus_info(nwb):
    """Get information about when stimuli were shown."""
    clip_intervals = nwb.intervals['Clip']

    clip_starts = np.array(clip_intervals.start_time[:])
    clip_stops = np.array(clip_intervals.stop_time[:])
    clip_types = np.array(clip_intervals.short_movie_name[:])

    return clip_starts, clip_stops, clip_types


# ============================================================================
# Feature extraction
# ============================================================================

def extract_neural_features(rs, timestamps, clip_starts, clip_stops, clip_types,
                           neuron_list, target_conditions, window_duration=None):
    """Extract mean neural activity for each stimulus presentation."""
    condition_names = {cond: i for i, cond in enumerate(target_conditions)}
    all_trials = []
    all_labels = []

    for i in range(len(clip_starts)):
        stim_type = clip_types[i]
        
        if stim_type not in target_conditions:
            continue

        start_time = clip_starts[i]
        stop_time = clip_stops[i] if window_duration is None else start_time + window_duration

        start_idx = np.searchsorted(timestamps, start_time)
        stop_idx = np.searchsorted(timestamps, stop_time)

        neural_chunk = np.array(rs.data[start_idx:stop_idx, neuron_list])
        neural_features = np.mean(neural_chunk, axis=0)

        all_trials.append(neural_features)
        all_labels.append(condition_names[stim_type])

    X = np.array(all_trials)
    y = np.array(all_labels)

    return X, y, condition_names


# ============================================================================
# Feature selection
# ============================================================================

def select_best_neurons(rs, timestamps, clip_starts, clip_stops, clip_types,
                       target_conditions, n_select=200):
    """Select the most informative neurons using ANOVA F-test."""
    from sklearn.feature_selection import f_classif
    
    all_neurons = list(range(rs.data.shape[1]))
    X_all, y_all, _ = extract_neural_features(
        rs, timestamps, clip_starts, clip_stops, clip_types,
        all_neurons, target_conditions, window_duration=None
    )

    f_scores, p_values = f_classif(X_all, y_all)
    best_indices = np.argsort(f_scores)[-n_select:]
    best_indices = sorted(best_indices.tolist())

    return best_indices


# ============================================================================
# Grid search
# ============================================================================

def grid_search_decoder(X, y, n_folds=5):
    """Use GridSearchCV to find best hyperparameters."""
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(random_state=42))
    ])

    param_grid = [
        {
            'classifier__C': [0.001, 0.01, 0.1, 1.0, 10, 100],
            'classifier__penalty': ['l2'],
            'classifier__solver': ['lbfgs', 'liblinear', 'saga'],
            'classifier__class_weight': [None, 'balanced'],
            'classifier__max_iter': [1000, 2000]
        },
        {
            'classifier__C': [0.001, 0.01, 0.1, 1.0, 10, 100],
            'classifier__penalty': ['l1'],
            'classifier__solver': ['liblinear', 'saga'],
            'classifier__class_weight': [None, 'balanced'],
            'classifier__max_iter': [1000, 2000]
        },
        {
            'classifier__penalty': [None],
            'classifier__solver': ['lbfgs', 'saga'],
            'classifier__class_weight': [None, 'balanced'],
            'classifier__max_iter': [1000, 2000]
        },
        {
            'classifier__C': [0.001, 0.01, 0.1, 1.0, 10, 100],
            'classifier__penalty': ['elasticnet'],
            'classifier__solver': ['saga'],
            'classifier__l1_ratio': [0.5],
            'classifier__class_weight': [None, 'balanced'],
            'classifier__max_iter': [2000]
        }
    ]

    cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=cv,
        scoring='accuracy',
        n_jobs=-1,  # uses cpu cores
        verbose=0,
        refit=True
    )

    grid_search.fit(X, y)

    return grid_search.best_estimator_, grid_search.best_params_, grid_search.best_score_


def evaluate_best_model(best_pipeline, X, y, n_folds=5):
    """Evaluate the best model with cross-validation."""
    from sklearn.base import clone
    
    cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    fold_accuracies = []

    for fold_idx, (train_idx, test_idx) in enumerate(cv.split(X, y)):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        pipeline = clone(best_pipeline)
        pipeline.fit(X_train, y_train)

        y_pred = pipeline.predict(X_test)
        accuracy = np.mean(y_test == y_pred)
        fold_accuracies.append(accuracy)

    return fold_accuracies


def analyze_single_plane(rs, rs_name, timestamps, clip_starts, clip_stops, 
                         clip_types, target_conditions, n_neurons=200, n_folds=5):
    """Analyze a single ROI response series (imaging plane)."""
    
    print(f"\n{'='*70}")
    print(f"Analyzing: {rs_name}")
    print(f"{'='*70}")
    
    start_time = time.time()
    
    # Get basic info
    n_total_neurons = rs.data.shape[1]
    n_timepoints = rs.data.shape[0]
    print(f"Total neurons in plane: {n_total_neurons}")
    print(f"Total timepoints: {n_timepoints}")
    
    # Select best neurons
    print(f"\nSelecting top {n_neurons} neurons using ANOVA F-score...")
    neuron_selection_start = time.time()
    neuron_list = select_best_neurons(
        rs, timestamps, clip_starts, clip_stops, clip_types,
        target_conditions, n_select=min(n_neurons, n_total_neurons)
    )
    neuron_selection_time = time.time() - neuron_selection_start
    print(f"  Selection completed in {neuron_selection_time:.1f}s")
    
    # Extract features
    print("Extracting neural features...")
    feature_start = time.time()
    X, y, condition_names = extract_neural_features(
        rs, timestamps, clip_starts, clip_stops, clip_types,
        neuron_list, target_conditions, window_duration=None
    )
    feature_time = time.time() - feature_start
    print(f"  Extracted {X.shape[0]} trials × {X.shape[1]} features in {feature_time:.1f}s")
    
    # Grid search
    print(f"Running grid search with {n_folds}-fold CV...")
    grid_start = time.time()
    best_pipeline, best_params, best_score = grid_search_decoder(X, y, n_folds=n_folds)
    grid_time = time.time() - grid_start
    print(f"  Grid search completed in {grid_time:.1f}s")
    
    # Final evaluation
    print("Evaluating best model...")
    eval_start = time.time()
    fold_accs = evaluate_best_model(best_pipeline, X, y, n_folds=n_folds)
    eval_time = time.time() - eval_start
    
    total_time = time.time() - start_time
    
    # Results
    print(f"\n{'='*70}")
    print("RESULTS")
    print(f"{'='*70}")
    print(f"Accuracy: {np.mean(fold_accs)*100:.2f}% ± {np.std(fold_accs)*100:.2f}%")
    print(f"\nBest parameters:")
    params_clean = {k.replace('classifier__', ''): v for k, v in best_params.items()}
    for param, value in params_clean.items():
        print(f"  {param}: {value}")
    print(f"\nTiming breakdown:")
    print(f"  Neuron selection: {neuron_selection_time:.1f}s")
    print(f"  Feature extraction: {feature_time:.1f}s")
    print(f"  Grid search: {grid_time:.1f}s")
    print(f"  Evaluation: {eval_time:.1f}s")
    print(f"  Total: {total_time:.1f}s")
    
    return {
        'plane_name': rs_name,
        'n_neurons': n_total_neurons,
        'n_selected': len(neuron_list),
        'n_trials': X.shape[0],
        'accuracy_mean': np.mean(fold_accs),
        'accuracy_std': np.std(fold_accs),
        'best_params': best_params,
        'fold_accuracies': fold_accs,
        'timing': {
            'neuron_selection': neuron_selection_time,
            'feature_extraction': feature_time,
            'grid_search': grid_time,
            'evaluation': eval_time,
            'total': total_time
        }
    }


# ============================================================================
# Main Analysis - Loop through all planes
# ============================================================================

print("="*70)
print("MULTI-PLANE NEURAL DECODING ANALYSIS")
print("="*70)

# Load data
print("\nConnecting to dataset...")
nwb = connect_to_data()

# Get all ROI response series
print("Finding all imaging planes...")
all_series = get_all_roi_response_series(nwb)
print(f"Found {len(all_series)} ROI response series:")
for name in all_series.keys():
    print(f"  - {name}")

# Get stimulus info (same for all planes)
clip_starts, clip_stops, clip_types = get_stimulus_info(nwb)
print(f"\nTotal stimulus presentations: {len(clip_starts)}")
unique_types = np.unique(clip_types)
for stim_type in unique_types:
    count = np.sum(clip_types == stim_type)
    print(f"  {stim_type}: {count} presentations")

# Analysis parameters
target_conditions = ['Cinematic', 'Rendered', 'sports1m']
n_neurons = 200
n_folds = 5

# Analyze each plane
results = []
total_start = time.time()

for rs_name, rs in all_series.items():
    try:
        timestamps = np.array(rs.timestamps[:100000])
        result = analyze_single_plane(
            rs, rs_name, timestamps, clip_starts, clip_stops, clip_types,
            target_conditions, n_neurons=n_neurons, n_folds=n_folds
        )
        results.append(result)
    except Exception as e:
        print(f"\n Error analyzing {rs_name}: {str(e)}")
        continue

total_time = time.time() - total_start

# ============================================================================
# Summary across all planes
# ============================================================================

print("\n" + "="*70)
print("SUMMARY ACROSS ALL PLANES")
print("="*70)

if results:
    # Sort by accuracy
    results_sorted = sorted(results, key=lambda x: x['accuracy_mean'], reverse=True)
    
    print(f"\nTotal planes analyzed: {len(results)}")
    print(f"Total time: {total_time/60:.1f} minutes")
    print(f"\nRanking by decoding accuracy:")
    print(f"{'Rank':<6} {'Plane':<20} {'Neurons':<10} {'Accuracy':<15} {'Time (s)'}")
    print("-"*70)
    
    for i, res in enumerate(results_sorted, 1):
        print(f"{i:<6} {res['plane_name']:<20} {res['n_neurons']:<10} "
              f"{res['accuracy_mean']*100:>5.2f}% ± {res['accuracy_std']*100:>4.2f}%  "
              f"{res['timing']['total']:>7.1f}")
    
    # Best and worst
    best = results_sorted[0]
    worst = results_sorted[-1]
    
    print(f"\n Best plane: {best['plane_name']}")
    print(f"   Accuracy: {best['accuracy_mean']*100:.2f}% ± {best['accuracy_std']*100:.2f}%")
    
    print(f"\n Worst plane: {worst['plane_name']}")
    print(f"   Accuracy: {worst['accuracy_mean']*100:.2f}% ± {worst['accuracy_std']*100:.2f}%")
    
    print(f"\n Accuracy range: {worst['accuracy_mean']*100:.2f}% - {best['accuracy_mean']*100:.2f}%")
    print(f"   Difference: {(best['accuracy_mean'] - worst['accuracy_mean'])*100:.2f} percentage points")

print("\n" + "="*70)

MULTI-PLANE NEURAL DECODING ANALYSIS

Connecting to dataset...
Finding all imaging planes...
Found 8 ROI response series:
  - RoiResponseSeries1
  - RoiResponseSeries2
  - RoiResponseSeries3
  - RoiResponseSeries4
  - RoiResponseSeries5
  - RoiResponseSeries6
  - RoiResponseSeries7
  - RoiResponseSeries8

Total stimulus presentations: 384
  Cinematic: 128 presentations
  Rendered: 128 presentations
  sports1m: 128 presentations

Analyzing: RoiResponseSeries1
Total neurons in plane: 643
Total timepoints: 40000

Selecting top 200 neurons using ANOVA F-score...
  Selection completed in 142.7s
Extracting neural features...
  Extracted 384 trials × 200 features in 8.4s
Running grid search with 5-fold CV...
  Grid search completed in 214.4s
Evaluating best model...

RESULTS
Accuracy: 65.87% ± 4.86%

Best parameters:
  C: 0.1
  class_weight: None
  max_iter: 1000
  penalty: l2
  solver: lbfgs

Timing breakdown:
  Neuron selection: 142.7s
  Feature extraction: 8.4s
  Grid search: 214.4s
  Eval